# PROGRAMACIÓN GENÉTICA

## Usaremos la librería DEAP
En pocas palabras, DEAP es una **caja de herramientas** (un *framework*) que te facilita enormemente la creación de algoritmos genéticos y otros algoritmos evolutivos en Python.

En lugar de que tengas que programar todo desde cero, DEAP te da las piezas ya hechas para que tú las armes. Principalmente se encarga de:

* **Crear y manejar la población** de individuos (tus posibles soluciones).
* **Implementar los operadores genéticos** más comunes: **selección**, **cruce** (*crossover*) y **mutación**. Ya vienen listos para usar.
* **Orquestar todo el ciclo evolutivo**, es decir, el bucle principal que repite el proceso de evaluación y evolución generación tras generación.

Básicamente, te permite concentrarte en lo más importante de tu problema específico:

1.  **Cómo representar una solución** (la estructura de los "genes").
2.  **Cómo medir qué tan buena es** cada solución (la función de *fitness*).

DEAP se encarga de toda la lógica pesada del algoritmo, ahorrándote un montón de tiempo y código. 🧬

In [20]:
#Importamos Librerias
import operator 
import random
import math
import numpy as np 
from deap import base, creator, gp, tools, algorithms


In [21]:
#EJERCICIO: Buscar ajustar una funcion cúbica

def funcion_objetivo(x): #definimos nuestra funcion objetivo
    return x**3 + x**2 + x + 1


In [22]:
#Creamos el conjunto de primitivas
pset = gp.PrimitiveSet ("MAIN",1) # Es el número de argumentos (x)
pset.addPrimitive(operator.add, 2) # suma
pset.addPrimitive(operator.sub, 2) # resta
pset.addPrimitive(operator.mul, 2)# multiplicación
pset.addPrimitive(operator.neg, 1)# negación (cambio de signo)
pset.addPrimitive(math.sin, 1)# seno
pset.addPrimitive(math.cos, 1)# coseno
pset.addEphemeralConstant ("rand101", lambda: random.uniform(-1, 1)) # constante aleatoria


c:\Users\mateo\AppData\Local\Programs\Python\Python313\Lib\site-packages\deap\gp.py:257: RuntimeWarning: Ephemeral rand101 function cannot be pickled because its generating function is a lambda function. Use functools.partial instead.
  warnings.warn("Ephemeral {name} function cannot be "


In [23]:
#RENOMBRAMOS EL ARGUMENTO
pset.renameArguments(ARG0='x')

#DEFINIR EL TIPO DE FITNESS (minimizar el error)
creator.create('FitnessMin',base.Fitness, weights = (-1.0,))
creator.create('Individuo',gp.PrimitiveTree, fitness = creator.FitnessMin)

#Funciones para inicializar individuos y población
toolbox = base.Toolbox()

toolbox.register('expr', gp.genHalfAndHalf, pset = pset, min_ = 1, max_ = 2)
toolbox.register('individuo', tools.initIterate, creator.Individuo, toolbox.expr)
toolbox.register('population', tools.initRepeat,list,toolbox.individuo)

# FUNCION PARA COMPILAR LOS ÁRBOLES EN FUNCIONES EJECUTABLES
toolbox.register("compile", gp.compile, pset=pset)

# FUNCIÓN DE EVALUACIÓN (error cuadrático medio)
# Puntos de datos
data_points = np.linspace(-1, 1, 20) # 20 puntos entre -1 y 1
valores_objetivo = []
for x in data_points:
    y = funcion_objetivo(x)
    valores_objetivo.append(y)

def evaluar(individual):
    # Convertir el individuo en una función
    func = toolbox.compile(expr=individual)
    
    # Calcular los errores cuadráticos
    errores_cuad = []
    for x, y in zip(data_points, valores_objetivo):
        error = (func(x) - y)**2
        errores_cuad.append(error)
        
    # Devolver el error cuadrático medio (MSE)
    return math.fsum(errores_cuad) / len(data_points),

#MECANISMOS EVOLUTIVOS
toolbox.register('evaluate', evaluar)
toolbox.register('select',tools.selTournament, tournsize = 3)
toolbox.register('mate',gp.cxOnePoint)
toolbox.register('expr_mut',gp.genFull,min_=0,max_=2)
toolbox.register('mutate',gp.mutUniform,expr=toolbox.expr_mut,pset=pset)

#SE LIMITA LA PROFUNDIDAD DE LOS INDIVIDUOS
toolbox.decorate('mate',gp.staticLimit(key=lambda ind: ind.height,max_value=17))
toolbox.decorate('mutate',gp.staticLimit(key=lambda ind: ind.height,max_value=17))

c:\Users\mateo\AppData\Local\Programs\Python\Python313\Lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'FitnessMin' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
c:\Users\mateo\AppData\Local\Programs\Python\Python313\Lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'Individuo' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [24]:
# ALGORITMO EVOLUTIVO
# Crear población inicial
Poblacion = toolbox.population(n=100)
hof = tools.HallOfFame(1)

# Estadísticas de la población
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

# Algoritmo evolutivo simple
algorithms.eaSimple(Poblacion, toolbox, 0.5, 0.05, 100, stats=stats, halloffame=hof, verbose=True)

# Mostrar el mejor individuo (la mejor expresión)
print("\nMejor individuo:")
print(hof[0])
print()

# PARA VISUALIZAR EL MEJOR INDIVIDUO
!pip install sympy
import sympy
x = sympy.Symbol("x")

# Mapeo de nombres usados en DEAP a funciones de sympy
replacements = {
    "add": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "mul": lambda a, b: a * b,
    "neg": lambda a: -a,
    "sin": sympy.sin,
    "cos": sympy.cos,
}

expr_str = str(hof[0]) # expresión en string que da DEAP
expr_sympy = sympy.sympify(expr_str, locals=replacements)

print("\nExpresión simplificada:\n", sympy.simplify(expr_sympy))

#Convertir el mejor individuo en una funcion y mostrar cómo ajusta los pts
best_func = toolbox.compile(expr=hof[0])
print()

#Comparar con los valores originales
for x,y in zip(data_points,valores_objetivo):
    print(f'x = {x:.2f},predicho = {best_func(x):.2f},real = {y:.2f}')

gen	nevals	avg    	std    	min     	max    
0  	100   	2.99686	1.72085	0.489216	8.35438
1  	53    	2.15417	1.22194	0.595513	8.93813
2  	52    	1.7834 	1.03203	0.316432	5.32059
3  	60    	1.59708	0.920615	0.316432	5.36209
4  	39    	1.18306	0.508809	0.316432	2.93139
5  	52    	1.14609	0.564592	0.316432	3.31808
6  	54    	1.17923	1.0734  	0.316432	7.15526
7  	52    	1.14521	0.998306	0.218621	5.59054
8  	45    	0.957459	0.945286	0.218621	5.02899
9  	65    	1.08317 	0.979469	0.197843	5.02899
10 	45    	0.80902 	0.890378	0.191686	5.02899
11 	53    	0.990855	1.23059 	0.191686	6.62126
12 	56    	0.768832	0.916744	0.191686	6.63628
13 	58    	0.923757	1.14739 	0.181067	6.43348
14 	42    	0.523009	0.540828	0.181067	2.07596
15 	62    	0.754424	0.819288	0.181067	4.3228 
16 	55    	0.931946	1.23066 	0.181067	7.07844
17 	56    	0.697949	0.909122	0.180164	5.81031
18 	57    	0.619921	0.848876	0.180164	4.46006
19 	56    	0.739733	0.927101	0.180164	4.73547
20 	58    	0.731541	0.771443	0.180164	3.18374
2